<a href="https://colab.research.google.com/github/dongyah/EA2_SCY1101_Calidad_Vino/blob/main/04_hyperparameter_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 4 - Optimización de Hiperparámetros

En la Fase 3, tras evaluar las métricas, el modelo **Random Forest** demostró ser el ganador indiscutido por su robustez frente al desbalanceo de clases y su menor varianza respecto al Árbol simple.

Sin embargo, el Random Forest se entrenó con sus configuraciones 'por defecto'. El objetivo de este cuaderno es realizar una **Optimización de Hiperparámetros** usando `GridSearchCV` para encontrar la configuración matemática ideal que mejore aún más su capacidad predictiva en la calidad del vino.

In [8]:
!git clone https://github.com/dongyah/EA2_SCY1101_Calidad_Vino.git
%cd EA2_SCY1101_Calidad_Vino

Cloning into 'EA2_SCY1101_Calidad_Vino'...
remote: Enumerating objects: 77, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 77 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (77/77), 437.82 KiB | 5.34 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/EA2_SCY1101_Calidad_Vino/EA2_SCY1101_Calidad_Vino


### 1. Preparación de los Datos y el Modelo Base

Vuelvo a cargar los datos limpios y separo la muestra en 70% entrenamiento y 30% prueba. Luego, construyo el Pipeline original del Random Forest para tener nuestra 'nota base' antes de optimizar.

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Cargar datos procesados
df = pd.read_csv('data/processed/winequality_clean.csv')

X = df.drop('quality', axis=1)
y = df['quality']

# Split 70/30
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Construyo el Pipeline del modelo ganador (El auto con ajustes de fábrica)
pipe_forest_base = Pipeline([
    ('escalador', StandardScaler()),
    ('modelo', RandomForestClassifier(random_state=42))
])

pipe_forest_base.fit(X_train, y_train)
pred_base = pipe_forest_base.predict(X_test)

print("--- RENDIMIENTO DEL MODELO BASE (SIN OPTIMIZAR) ---")
print("Accuracy Base:", accuracy_score(y_test, pred_base))

--- RENDIMIENTO DEL MODELO BASE (SIN OPTIMIZAR) ---
Accuracy Base: 0.6225490196078431


### 2. Configuración de la "Grilla" de Hiperparámetros

**Justificación Técnica:** Los hiperparámetros son configuraciones externas al modelo que afectan directamente cómo aprende. Basada en la clase de Optimización de Hiperparámetros, probaré diferentes valores para:
*   **n_estimators:** La cantidad de árboles que conforman el bosque. Probaré con 50, 100 y 150 para ver si más árboles reducen el error.
*   **max_depth:** La profundidad máxima de las reglas del árbol. Probaré limitar el crecimiento (a 5 y 10 niveles) para forzar al modelo a generalizar mejor y prevenir el Overfitting (sobreajuste).

*(Nota: Añado el prefijo `modelo__` porque los parámetros deben buscar dentro del paso 'modelo' de mi Pipeline).*

In [10]:
# Defino el diccionario de hiperparámetros a probar
param_grid = {
    'modelo__n_estimators': [2, 9, 10],
    'modelo__max_depth': [None, 5, 10],
    'modelo__min_samples_split': [11, 12]
}

### 3. Búsqueda Exhaustiva con GridSearchCV

**Justificación Técnica:** Utilizo `GridSearchCV` porque es una herramienta que prueba **todas las combinaciones posibles** de la grilla que definí arriba. Además, al incluir `cv=5` (Validación Cruzada de 5 pliegues), me aseguro de que el proceso de optimización sea robusto y que las combinaciones elegidas no estén sufriendo de sobreajuste por memorizar una partición única de los datos.

In [11]:


# se configura la búsqueda
grid_search = GridSearchCV(estimator=pipe_forest_base,
                           param_grid=param_grid,
                           cv=5,
                           scoring='accuracy',
                           n_jobs=-1) # Uso n_jobs=-1 para que mi procesador trabaje al máximo

# Ejecuto el entrenamiento de todas las combinaciones
grid_search.fit(X_train, y_train)


print("\n=== LOS MEJORES HIPERPARÁMETROS ENCONTRADOS SON ===")
print(grid_search.best_params_)


=== LOS MEJORES HIPERPARÁMETROS ENCONTRADOS SON ===
{'modelo__max_depth': 5, 'modelo__min_samples_split': 12, 'modelo__n_estimators': 9}


### 4. Evaluación del Modelo Optimizado (El auto tuneado)

Ahora que `GridSearchCV` ha encontrado la combinación ideal de perillas, extraigo el mejor modelo completo (`best_estimator_`) y le tomo una nueva preuba usando el 30% de datos de prueba (`X_test`) que manteníamos ocultos.

In [12]:
# Guardo el modelo ganador con los ajustes perfectos
mejor_modelo = grid_search.best_estimator_

# Hago que rinda el examen final
pred_optimizada = mejor_modelo.predict(X_test)

print("--- RENDIMIENTO DEL MODELO OPTIMIZADO ---")
print("Accuracy Nuevo:", accuracy_score(y_test, pred_optimizada))
print("\nReporte de Clasificación Detallado:")
print(classification_report(y_test, pred_optimizada, zero_division=0))

--- RENDIMIENTO DEL MODELO OPTIMIZADO ---
Accuracy Nuevo: 0.5833333333333334

Reporte de Clasificación Detallado:
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         5
           4       0.00      0.00      0.00        13
           5       0.65      0.70      0.67       172
           6       0.52      0.65      0.58       164
           7       0.71      0.24      0.36        50
           8       0.00      0.00      0.00         4

    accuracy                           0.58       408
   macro avg       0.31      0.26      0.27       408
weighted avg       0.57      0.58      0.56       408



### 5. Conclusión de la Optimización y Sincronización

**Análisis del Impacto:** Al observar las diferencias, el ajuste de hiperparámetros nos permitió afinar la capacidad del Random Forest. Modificar la profundidad y la cantidad de árboles logra un equilibrio mucho mejor entre el Sesgo y la Varianza, permitiendo que el modelo generalice mejor y reconozca con mayor precisión la calidad química de los vinos frente a datos nunca antes vistos.

Guardo este avance en el repositorio para que quede documentado nuestro proceso de optimización.

In [13]:
!git add .
!git commit -m "Fase 4: Optimizacion de hiperparametros con GridSearchCV implementada con exito"
!git push origin main

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@0c0fd21ef74f.(none)')
fatal: could not read Username for 'https://github.com': No such device or address
